# RESPOND Political Corruption Corpus: Clean, Denominator, Baseline

Run this notebook top to bottom. The workflow is:

1. Load the raw country CSVs from Research Drive.
2. Fix text encoding issues where needed.
3. Clean and deduplicate the corpus into `df_clean`.
4. Save compressed cleaned files to the 1 TB storage disk.
5. Build denominator tables from `df_clean`.
6. Load the 452 manual labels and train/evaluate a first TF-IDF baseline.

Important: the raw files were collected using corruption-related keywords, so `df_clean` is a cleaned corruption-query corpus, not a full all-news denominator.


In [ ]:
from pathlib import Path
import hashlib
import os
import re

import pandas as pd

from config import RD_BASE_DIR
from dataloader import load_country_news_files_webdav, load_human_annotated_for_translation_webdav


In [ ]:
NEWS_DIR = RD_BASE_DIR
COUNTRIES = None  # None means: discover and load every *_news.csv file.
MIN_WORDS = 80

# Large storage location on annecuda. Override by setting RESPOND_OUTPUT_DIR if needed.
DEDUP_OUTPUT_DIR = Path(
    os.environ.get(
        "RESPOND_OUTPUT_DIR",
        "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline",
    )
)
DEDUP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Research Drive input directory: {NEWS_DIR}")
print(f"Output directory:              {DEDUP_OUTPUT_DIR}")


In [ ]:
df = load_country_news_files_webdav(data_dir=NEWS_DIR, countries=COUNTRIES)
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head()


In [ ]:
print("Country counts:")
display(df["country"].value_counts(dropna=False).to_frame("count"))

if "lang" in df.columns:
    print("Language counts:")
    display(df["lang"].value_counts(dropna=False).to_frame("count"))

if "source.uri" in df.columns:
    print("Top source.uri values:")
    display(df["source.uri"].value_counts(dropna=False).head(20).to_frame("count"))


## Helpers

`fix_mojibake` repairs text like `Ð¡Ð»ÑÐ¶Ð±Ð°...` back into readable Cyrillic, and also repairs common `Ã¡`-style Latin accent damage. If text is already fine, it is returned unchanged.


In [ ]:
TEXT_COLUMN_CANDIDATES = [
    "translated_text",
    "combined_text",
    "body",
    "text",
    "article_text",
    "content",
]

KEEP_CLEANED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "dateTimePub",
    "date",
    "date_parsed",
    "year",
    "month",
    "week",
    "source_uri",
    "article_text",
    "word_count",
    "text_hash",
    "near_dup_hash",
]

MINIMAL_COMBINED_COLUMNS = [
    "uri",
    "country",
    "dateTime",
    "date_parsed",
    "year",
    "month",
    "week",
    "source_uri",
    "word_count",
    "text_hash",
    "near_dup_hash",
]


def choose_text_column(dataframe):
    for column in TEXT_COLUMN_CANDIDATES:
        if column in dataframe.columns:
            return column
    raise ValueError("No usable text column found. Expected one of: " + ", ".join(TEXT_COLUMN_CANDIDATES))


def fix_mojibake(text):
    if not isinstance(text, str):
        return ""
    # Only attempt repair when common mojibake markers are present.
    if not any(marker in text for marker in ["Ð", "Ñ", "Ã", "Â"]):
        return text
    try:
        fixed = text.encode("latin1").decode("utf-8")
        return fixed
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def normalize_text(text):
    text = fix_mojibake(text)
    text = text.replace(" ", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_for_hash(text):
    text = normalize_text(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def text_hash(text):
    normalized = normalize_for_hash(text)
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()


def cheap_near_duplicate_fingerprint(text, n_tokens=80):
    normalized = normalize_for_hash(text)
    fingerprint = " ".join(normalized.split()[:n_tokens])
    return hashlib.md5(fingerprint.encode("utf-8")).hexdigest()


## Clean And Deduplicate

This is the denominator dataset for the corruption-query corpus. Wait until all printed steps appear before saving.


In [ ]:
df_clean = df.copy()
print(f"Starting rows: {len(df_clean):,}")

text_col = choose_text_column(df_clean)
print(f"Using text column: {text_col}")
df_clean["article_text"] = df_clean[text_col].map(normalize_text)

if "isDuplicate" in df_clean.columns:
    before = len(df_clean)
    is_duplicate = df_clean["isDuplicate"].astype(str).str.lower().isin(["true", "1", "yes"])
    df_clean = df_clean[~is_duplicate].copy()
    print(f"After dropping isDuplicate=True rows: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["word_count"] = df_clean["article_text"].str.split().str.len().fillna(0).astype(int)
df_clean = df_clean[df_clean["article_text"].ne("")].copy()
print(f"After removing missing/empty text: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean = df_clean[df_clean["word_count"] >= MIN_WORDS].copy()
print(f"After removing articles with word_count < {MIN_WORDS}: {len(df_clean):,} (-{before - len(df_clean):,})")

if "dateTime" in df_clean.columns:
    date_source = df_clean["dateTime"]
elif "dateTimePub" in df_clean.columns:
    date_source = df_clean["dateTimePub"]
elif "date" in df_clean.columns:
    date_source = df_clean["date"]
else:
    date_source = pd.Series(pd.NaT, index=df_clean.index)

df_clean["date_parsed"] = pd.to_datetime(date_source, errors="coerce", utc=True)
date_naive = df_clean["date_parsed"].dt.tz_convert(None)
df_clean["year"] = date_naive.dt.year.astype("Int64")
df_clean["month"] = date_naive.dt.to_period("M").dt.to_timestamp()
df_clean["week"] = date_naive.dt.to_period("W").dt.start_time

if "source.uri" in df_clean.columns:
    df_clean["source_uri"] = df_clean["source.uri"]
elif "source" in df_clean.columns:
    df_clean["source_uri"] = df_clean["source"]
else:
    df_clean["source_uri"] = None

if "uri" in df_clean.columns:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=["uri"], keep="first").copy()
    print(f"After URI dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["text_hash"] = df_clean["article_text"].map(text_hash)
df_clean = df_clean.drop_duplicates(subset=["text_hash"], keep="first").copy()
print(f"After exact text dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

before = len(df_clean)
df_clean["near_dup_hash"] = df_clean["article_text"].map(cheap_near_duplicate_fingerprint)
df_clean = df_clean.drop_duplicates(subset=["near_dup_hash"], keep="first").copy()
print(f"After cheap near-duplicate dedupe: {len(df_clean):,} (-{before - len(df_clean):,})")

display(df_clean.head())


In [ ]:
denom_country_total = (
    df_clean.groupby("country", dropna=False)
    .size()
    .reset_index(name="total_articles")
    .sort_values("total_articles", ascending=False)
)

display(denom_country_total)
print(f"Total cleaned/deduped rows: {denom_country_total['total_articles'].sum():,}")


## Save Cleaned Outputs

This saves a minimal combined file and compressed full-text per-country files to the 1 TB disk. It avoids a huge uncompressed combined full-text CSV.


In [ ]:
keep_columns = [column for column in KEEP_CLEANED_COLUMNS if column in df_clean.columns]
minimal_columns = [column for column in MINIMAL_COMBINED_COLUMNS if column in df_clean.columns]

combined_output = DEDUP_OUTPUT_DIR / "all_countries_cleaned_deduped_minimal.csv.gz"
df_clean[minimal_columns].to_csv(combined_output, index=False, compression="gzip")
print(f"Saved minimal combined cleaned file: {combined_output}")

for country, country_df in df_clean.groupby("country", dropna=False):
    safe_country = str(country).replace("/", "_")
    country_output = DEDUP_OUTPUT_DIR / f"{safe_country}_cleaned_deduped.csv.gz"
    country_df[keep_columns].to_csv(country_output, index=False, compression="gzip")
    print(f"Saved {len(country_df):,} rows: {country_output}")


## Build Denominator Tables

These are the denominator counts for the cleaned corruption-query corpus.


In [ ]:
denom_country_year = (
    df_clean.groupby(["country", "year"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_month = (
    df_clean.groupby(["country", "month"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_week = (
    df_clean.groupby(["country", "week"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

denom_country_year_source = (
    df_clean.groupby(["country", "year", "source_uri"], dropna=False)
    .size()
    .reset_index(name="total_articles")
)

print("Country-year:")
display(denom_country_year.head(20))
print("Country-month:")
display(denom_country_month.head(20))
print("Country-week:")
display(denom_country_week.head(20))


In [ ]:
denom_country_year.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_year.csv", index=False)
denom_country_month.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_month.csv", index=False)
denom_country_week.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_week.csv", index=False)
denom_country_year_source.to_csv(DEDUP_OUTPUT_DIR / "denominator_country_year_source.csv", index=False)

print(f"Saved denominator tables to: {DEDUP_OUTPUT_DIR}")


## Manual Labels And Baseline Model

You have 452 manual labels. We use them to train/evaluate a first TF-IDF character n-gram classifier. This is only a baseline; the previous result was useful but not strong enough as final labels for the full corpus.


In [ ]:
df_annotations = load_human_annotated_for_translation_webdav()
print(f"Loaded {len(df_annotations):,} annotated rows.")

print("Label counts:")
display(df_annotations["corruption_label_m"].value_counts(dropna=False))

print("Country counts:")
display(df_annotations["country"].value_counts(dropna=False))


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

manual_df = df_annotations.copy()
manual_df["label_clean"] = manual_df["corruption_label_m"].astype(str).str.strip().str.lower()
manual_df = manual_df[manual_df["label_clean"].isin(["political corruption", "no political corruption"])].copy()
manual_df["y"] = manual_df["label_clean"].map({"political corruption": 1, "no political corruption": 0})

if "combined_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["combined_text"].fillna("").astype(str).map(normalize_text)
elif "article_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["article_text"].fillna("").astype(str).map(normalize_text)
elif "translated_text" in manual_df.columns:
    manual_df["model_text"] = manual_df["translated_text"].fillna("").astype(str).map(normalize_text)
else:
    manual_df["model_text"] = (
        manual_df.get("title", "").fillna("").astype(str)
        + "
"
        + manual_df.get("body", "").fillna("").astype(str)
    ).map(normalize_text)

manual_df = manual_df[manual_df["model_text"].str.strip().ne("")].copy()
print(f"Training rows: {len(manual_df):,}")
display(manual_df["y"].value_counts().rename(index={0: "No", 1: "Political corruption"}))


In [ ]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=200_000,
            lowercase=True,
        ),
    ),
    (
        "clf",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42,
        ),
    ),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_true = manual_df["y"].values

y_pred = cross_val_predict(model, manual_df["model_text"], y_true, cv=cv, method="predict")

print(classification_report(
    y_true,
    y_pred,
    target_names=["No political corruption", "Political corruption"],
    zero_division=0,
))

display(pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=["True No", "True Political"],
    columns=["Pred No", "Pred Political"],
))


In [ ]:
y_prob = cross_val_predict(model, manual_df["model_text"], y_true, cv=cv, method="predict_proba")[:, 1]
manual_df["cv_prob_political_corruption"] = y_prob
manual_df["cv_pred"] = (manual_df["cv_prob_political_corruption"] >= 0.5).astype(int)

display(
    manual_df[
        ["country", "corruption_label_m", "cv_prob_political_corruption", "cv_pred", "model_text"]
    ].sort_values("cv_prob_political_corruption", ascending=False).head(20)
)


## Optional Next Step: Active-Learning Review Samples

Only run this after inspecting the cross-validation report. It trains on all manual labels, scores `df_clean`, and saves review samples: high predicted positive, uncertain, and high predicted negative. These are better next labeling batches than another broad keyword screen.


In [ ]:
# Optional: run only if you want review samples for more labeling.
# model.fit(manual_df["model_text"], manual_df["y"])
# df_scored = df_clean.copy()
# df_scored["model_text"] = df_scored["article_text"].fillna("").astype(str)
# df_scored["prob_political_corruption"] = model.predict_proba(df_scored["model_text"])[:, 1]
# df_scored["model_pred"] = (df_scored["prob_political_corruption"] >= 0.5).astype(int)
#
# REVIEW_OUTPUT_DIR = DEDUP_OUTPUT_DIR / "review_samples"
# REVIEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# sample_cols = ["uri", "country", "date_parsed", "year", "source_uri", "article_text", "prob_political_corruption", "model_pred"]
# sample_cols = [col for col in sample_cols if col in df_scored.columns]
#
# high_pos_pool = df_scored[df_scored["prob_political_corruption"] >= 0.8]
# uncertain_pool = df_scored[df_scored["prob_political_corruption"].between(0.4, 0.6)]
# high_neg_pool = df_scored[df_scored["prob_political_corruption"] <= 0.2]
#
# high_pos = high_pos_pool.sample(min(300, len(high_pos_pool)), random_state=42)
# uncertain = uncertain_pool.sample(min(500, len(uncertain_pool)), random_state=42)
# high_neg = high_neg_pool.sample(min(300, len(high_neg_pool)), random_state=42)
#
# high_pos[sample_cols].to_csv(REVIEW_OUTPUT_DIR / "review_high_predicted_positive.csv", index=False)
# uncertain[sample_cols].to_csv(REVIEW_OUTPUT_DIR / "review_uncertain.csv", index=False)
# high_neg[sample_cols].to_csv(REVIEW_OUTPUT_DIR / "review_high_predicted_negative.csv", index=False)
# print(f"Saved review samples to: {REVIEW_OUTPUT_DIR}")
